# Notebook 02 — Goodreads Shelf Scraper

## Overview
Scrapes non-western fantasy books from 10 Goodreads community shelves using Selenium.

### Why Goodreads?
- Community-curated shelves are highly relevant and well-tagged
- Books come with rich descriptions, ratings and cover images
- Shelves like `afrofuturism` and `asian-fantasy` are maintained by dedicated communities

### Why Selenium?
Goodreads uses JavaScript rendering and requires login for full shelf access.

**Output:** `Data/Raw/non_western_fantasy/goodreads_with_descriptions.csv`

> **Note:** Requires a Goodreads account and Chrome installed.

## 1. Setup

In [1]:
import os
import re
import time
import json
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import yaml

load_dotenv()

def find_repo_root():
    current = Path().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "config.yaml").exists():
            return parent
    raise FileNotFoundError("config.yaml not found")

REPO_ROOT = find_repo_root()
with open(REPO_ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

RAW_DIR     = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"
RAW_DIR.mkdir(parents=True, exist_ok=True)

GR_OUT      = REPO_ROOT / config["output_data_raw_non_western_fantasy"]["gr_results"]
OUTPUT_PATH = RAW_DIR / "goodreads_raw.csv"
DESC_CKPT   = RAW_DIR / "goodreads_desc_checkpoint.csv"

DELAY      = 3.0
DESC_DELAY = 2.0
SAVE_EVERY = 50

SHELVES = [
    "african-fantasy", "asian-fantasy", "indigenous-fantasy",
    "south-american-fantasy", "australian-fantasy", "middle-eastern-fantasy",
    "latin-american-fantasy", "afrofuturism", "african-science-fiction",
    "asian-science-fiction",
]

print(f"Repo root: {REPO_ROOT}")
print(f"Output:    {GR_OUT}")
print(f"Shelves:   {len(SHELVES)}")

# ── Guard: skip if data already exists ───────────────────────────────────────
if GR_OUT.exists():
    df_existing = pd.read_csv(GR_OUT)
    print(f"\nData already exists - {len(df_existing):,} books")
    print("Skipping scraping. Delete goodreads_with_descriptions.csv to re-scrape.")
    raise SystemExit("Done - data already collected")

Repo root: C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations
Output:    C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Raw\non_western_fantasy\goodreads_with_descriptions.csv
Shelves:   10

Data already exists - 2,129 books
Skipping scraping. Delete goodreads_with_descriptions.csv to re-scrape.


SystemExit: Done - data already collected

c:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. Browser Setup & Login

We launch Chrome with anti-detection flags. If credentials are in `.env`, login is automatic. Otherwise a manual login prompt will appear.

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_argument("--window-size=1280,900")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
driver.execute_script(
    "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
)
wait = WebDriverWait(driver, 20)
print("Browser launched")

Browser launched


In [3]:
GR_EMAIL    = os.getenv("GOODREADS_EMAIL", "")
GR_PASSWORD = os.getenv("GOODREADS_PASSWORD", "")

driver.get("https://www.goodreads.com/user/sign_in")
time.sleep(3)

if GR_EMAIL and GR_PASSWORD:
    try:
        wait.until(EC.element_to_be_clickable(
            (By.XPATH, "//a[contains(text(),'Sign in with email')]")
        )).click()
        time.sleep(2)
        wait.until(EC.presence_of_element_located((By.ID, "user_email"))).send_keys(GR_EMAIL)
        driver.find_element(By.ID, "user_password").send_keys(GR_PASSWORD)
        driver.find_element(By.NAME, "next").click()
        time.sleep(3)
        print("Login successful" if "sign_in" not in driver.current_url else "Check login manually")
    except Exception as e:
        print(f"Auto-login failed: {e} — please log in manually")
else:
    print("No credentials in .env — please log in manually in the browser window")

Auto-login failed: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0xbeb373+10553]
	chromedriver!GetHandleVerifier [0xbeb4a4+10684]
	chromedriver!(No symbol) [0x9f1e10]
	chromedriver!(No symbol) [0xa3a9da]
	chromedriver!(No symbol) [0xa3ac7b]
	chromedriver!(No symbol) [0xa7d162]
	chromedriver!(No symbol) [0xa5d8b4]
	chromedriver!(No symbol) [0xa7aa36]
	chromedriver!(No symbol) [0xa5d606]
	chromedriver!(No symbol) [0xa30039]
	chromedriver!(No symbol) [0xa30df4]
	chromedriver!GetHandleVerifier [0xe6ddb4+292f94]
	chromedriver!GetHandleVerifier [0xe6937d+28e55d]
	chromedriver!GetHandleVerifier [0xe88215+2ad3f5]
	chromedriver!GetHandleVerifier [0xc04c18+29df8]
	chromedriver!GetHandleVerifier [0xc0c6fd+318dd]
	chromedriver!GetHandleVerifier [0xbf3c78+18e58]
	chromedriver!GetHandleVerifier [0xbf3e42+19022]
	chromedriver!GetHandleVerifier [0xbdd45f+263f]
	KERNEL32!BaseThreadInitThunk [0x771f5d49+19]
	ntdll!RtlInitializeExceptionChain [0x7742d83b+6b]
	ntdll!RtlGetAppContainerNamedObjectP

In [4]:
# Verify login
if "sign_in" not in driver.current_url:
    print("Logged in — ready to scrape")
else:
    print("Not logged in yet — please log in manually and re-run this cell")

Logged in — ready to scrape


## 3. Parsing & Scraping Functions

In [5]:
def parse_shelf_page(page_source, shelf):
    soup    = BeautifulSoup(page_source, "html.parser")
    records = []
    for div in soup.select("div.elementList"):
        title_tag  = div.select_one("a.bookTitle")
        author_tag = div.select_one("span[itemprop='name']")
        grey_tag   = div.select_one("span.greyText.smallText")
        cover_tag  = div.select_one("img")
        if not title_tag: continue
        grey_text    = grey_tag.get_text(strip=True) if grey_tag else ""
        rating_m     = re.search(r'avg rating ([\d.]+)', grey_text)
        num_rating_m = re.search(r'([\d,]+)\s+ratings', grey_text)
        year_m       = re.search(r'published (\d{4})', grey_text)
        records.append({
            "title":          title_tag.get_text(strip=True),
            "author":         author_tag.get_text(strip=True) if author_tag else "",
            "goodreads_url":  f"https://www.goodreads.com{title_tag['href'].split('?')[0]}",
            "cover_url":      cover_tag["src"] if cover_tag and cover_tag.get("src") else "",
            "avg_rating":     float(rating_m.group(1)) if rating_m else None,
            "num_ratings":    int(num_rating_m.group(1).replace(",", "")) if num_rating_m else None,
            "year_published": int(year_m.group(1)) if year_m else None,
            "shelf":          shelf,
        })
    return records

def fetch_description(goodreads_url):
    driver.get(goodreads_url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    desc = (soup.select_one("div[data-testid='description'] span.Formatted") or
            soup.select_one("#description span") or
            soup.select_one("span.Formatted"))
    return desc.get_text(separator=" ", strip=True) if desc else ""

def scrape_shelf(shelf, max_pages=20):
    all_records = []
    driver.get(f"https://www.goodreads.com/shelf/show/{shelf}")
    time.sleep(3)
    page = 1
    while page <= max_pages:
        records = parse_shelf_page(driver.page_source, shelf)
        if not records: break
        all_records.extend(records)
        print(f"  Page {page}: {len(records)} books (total: {len(all_records)})")
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "a.next_page")
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(DELAY)
            page += 1
        except Exception:
            break
    return all_records

print("Functions defined")

Functions defined


## 4. Scrape All Shelves

In [6]:
all_records = []
for shelf in SHELVES:
    print(f"\n-- Shelf: {shelf} --")
    try:
        records = scrape_shelf(shelf, max_pages=20)
        print(f"   {len(records)} books")
        all_records.extend(records)
    except Exception as e:
        print(f"   Failed: {e}")
    time.sleep(DELAY)

df = pd.DataFrame(all_records)
df = df.drop_duplicates(subset=["title", "author"]).reset_index(drop=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nScraping complete - {len(df):,} books")
print(df["shelf"].value_counts().to_string())


-- Shelf: african-fantasy --
  Page 1: 50 books (total: 50)
  Page 2: 50 books (total: 100)
  Page 3: 50 books (total: 150)
  Page 4: 50 books (total: 200)
  Page 5: 21 books (total: 221)
   221 books

-- Shelf: asian-fantasy --
  Page 1: 50 books (total: 50)
  Page 2: 50 books (total: 100)
  Page 3: 50 books (total: 150)
  Page 4: 50 books (total: 200)
  Page 5: 50 books (total: 250)
  Page 6: 50 books (total: 300)
  Page 7: 50 books (total: 350)
  Page 8: 50 books (total: 400)
  Page 9: 50 books (total: 450)
  Page 10: 50 books (total: 500)
  Page 11: 50 books (total: 550)
  Page 12: 50 books (total: 600)
  Page 13: 50 books (total: 650)
  Page 14: 50 books (total: 700)
  Page 15: 50 books (total: 750)
  Page 16: 50 books (total: 800)
  Page 17: 50 books (total: 850)
  Page 18: 50 books (total: 900)
  Page 19: 50 books (total: 950)
  Page 20: 50 books (total: 1000)
   1000 books

-- Shelf: indigenous-fantasy --
  Page 1: 31 books (total: 31)
   31 books

-- Shelf: south-american-fan

## 5. Description Enrichment

Visits each book page individually to fetch the full description. Checkpointed every 50 books.

In [8]:
if DESC_CKPT.exists():
    df_desc   = pd.read_csv(DESC_CKPT)
    done_urls = set(df_desc.loc[
        df_desc["description"].notna() & (df_desc["description"] != ""),
        "goodreads_url"
    ])
    print(f"Resuming - {len(done_urls):,} already done")
else:
    df_desc           = df.copy()
    df_desc["description"] = ""
    done_urls         = set()

todo = df_desc[~df_desc["goodreads_url"].isin(done_urls)]
print(f"Books to fetch: {len(todo):,}")

for i, (idx, row) in enumerate(todo.iterrows(), 1):
    try:
        desc = fetch_description(row["goodreads_url"])
        df_desc.at[idx, "description"] = desc
        status = "ok" if desc else "empty"
    except Exception:
        df_desc.at[idx, "description"] = ""
        status = "error"
    if i % 10 == 0:
        print(f"  [{i}/{len(todo)}] {status} - {row['title'][:40]}")
    if i % SAVE_EVERY == 0:
        df_desc.to_csv(DESC_CKPT, index=False)
    time.sleep(DESC_DELAY)

df_desc.to_csv(GR_OUT, index=False)
print(f"\nDone - {len(df_desc):,} books")
print(f"Has description: {(df_desc['description'] != '').sum():,}")

Books to fetch: 2,129
  [10/2129] ok - Skin of the Sea (Skin of the Sea, #1)
  [20/2129] ok - The Famished Road (Paperback)
  [30/2129] ok - The Deep (Hardcover)
  [40/2129] ok - Immortal Dark (Immortal Dark Trilogy, #1
  [50/2129] ok - Moon Witch, Spider King (The Dark Star T
  [60/2129] ok - Blood Scion (Blood Scion #1)
  [70/2129] ok - The City of Brass (The Daevabad Trilogy,
  [80/2129] ok - A Song of Legends Lost (Invoker Trilogy,
  [90/2129] ok - A Practical Guide to Evil I (Kindle Edit
  [100/2129] ok - Onyeka and the Heroes of the Dawn (Onyek
  [110/2129] ok - Tales of Nevèrÿon (Return to Nevèrÿon, #
  [120/2129] ok - Hoodoo Heaven (Boudin, Barbecue, and Hoo
  [130/2129] ok - Forging a Nightmare (Paperback)
  [140/2129] ok - Koku Akanbi and the Heart of Midnight (J
  [150/2129] ok - Blood Like Magic (Blood Like Magic, #1)
  [160/2129] ok - The Blood Trials (The Blood Gift Duology
  [170/2129] ok - The Angel of Khan el-Khalili (Dead Djinn
  [180/2129] ok - A ​Queen of Gilded Hor

In [9]:
driver.quit()
print("Browser closed")

Browser closed


## Conclusions

- Goodreads descriptions are higher quality than Open Library descriptions
- Community shelf curation catches books that API queries miss
- The scraper is slow (~2 seconds per book) but produces the richest dataset
- Some noise (coloring books, academic texts) can appear on community shelves — filtered in Notebook 03

**Next:** Notebook 03 merges this data with Open Library and produces the final clean dataset.